# Ara25 — AraT5

Public research-preview notebook. Outputs and volatile metadata were removed. Set local dataset/output paths in the configuration cells before running.


Reveiw Link:

https://colab.research.google.com/drive/1bSYgolmdLqhLQGsXHB7TR5UQIIpOPRuy?usp=sharing

In [ ]:
!pip install -U transformers datasets rouge-score nltk

In [ ]:
!pip install -q transformers datasets evaluate rouge-score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q evaluate

In [ ]:
!pip install -q rouge-score

In [ ]:
!pip install -q sacrebleu

In [ ]:
# === AraT5: تهيئة صحيحة + تدريب 8 عصور (بدون توقف مبكر) مع توافق تلقائي مع نسخ Transformers ===
import os, json, re, random, gc, math, pandas as pd, numpy as np, torch
from datetime import datetime
from datasets import Dataset
from transformers import (
    T5TokenizerFast, T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, GenerationConfig, set_seed
)
from inspect import signature
import evaluate

# ---------------- Run config ----------------
MODEL_FAMILY = "arat5"
MODEL_NAME_OR_PATH = "UBC-NLP/AraT5-base"  # غيّره لو عندك تشيكبوينت آخر
RUN_NAME = f"{MODEL_FAMILY}-base-{datetime.now().strftime('%Y%m%d-%H%M')}"
OUT = f"/content/drive/MyDrive/sum_runs/{RUN_NAME}"
for sub in ["checkpoints", "logs", "preds", "metrics", "artifacts"]:
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)

try:
    import transformers as _tf; print("🤗 Transformers version:", _tf.__version__)
except Exception:
    pass
print("RUN_NAME:", RUN_NAME)
print("OUT:", OUT)
print("BASE:", MODEL_NAME_OR_PATH)

# ---------------- System / Seeds ----------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True
if torch.cuda.is_available():
    try: torch.cuda.empty_cache()
    except: pass

SEED = 42
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- Data ----------------
JSONL = "/content/drive/MyDrive/PACKED2x2_6144/train_ready_6144.jsonl"  # ثبّت المسار لو مختلف
AR_PREFIX = "لخص بالعربية الفصحى بدقة واختصار ووضوح:"
MAX_IN, MAX_OUT = 6144, 512

with open(JSONL, "r", encoding="utf-8") as f:
    recs = [json.loads(l) for l in f if l.strip()]
assert recs, f"❌ لا توجد سجلات في {JSONL}"

df = pd.DataFrame(recs)
ds = Dataset.from_pandas(df, preserve_index=False)

# تأكّد من وجود عمود الهدف
TARGET_KEYS = ["target_summary", "summary", "target"]
tgt_key = next((k for k in TARGET_KEYS if k in ds.column_names), None)
assert tgt_key is not None, f"❌ لم أجد عمود الملخص الهدف. جرّبت: {TARGET_KEYS}"

# ---------------- Cleaning helpers ----------------
_dup_open_re = re.compile(r'(<ARTICLE_(\d+)>)\s*(?:\1)+')
def clean_merged(s: str) -> str:
    return _dup_open_re.sub(r'\1', s or "")

def ensure_prefix(s: str) -> str:
    s = (s or "").lstrip()
    return s if s.startswith(AR_PREFIX) else (AR_PREFIX + " " + s)

def _tidy(s: str) -> str:
    if not s: return s
    s = re.sub(r"\s+", " ", s)
    s = s.replace(" ،","،").replace(" .",".")
    return s.strip()

# ---------------- Tokenizer / Model (T5 فقط) ----------------
tok = T5TokenizerFast.from_pretrained(MODEL_NAME_OR_PATH, use_fast=True)
mdl = T5ForConditionalGeneration.from_pretrained(MODEL_NAME_OR_PATH)

# IDs الحرِجة
if getattr(mdl.config, "pad_token_id", None) is None:
    mdl.config.pad_token_id = tok.pad_token_id
if getattr(mdl.config, "decoder_start_token_id", None) is None:
    # لِـ T5 عادة نستخدم pad كقيمة dec_start الافتراضية
    mdl.config.decoder_start_token_id = tok.pad_token_id
mdl.config.use_cache = False  # مطلوب مع gradient_checkpointing

try: mdl.gradient_checkpointing_enable()
except Exception: pass

mdl = mdl.to(device)

# ---------------- Tokenization (HF dict-of-lists) ----------------
SRC_CANDIDATES = ["Merged_Articles", "merged_articles", "model_input", "text"]
src_key = next((c for c in SRC_CANDIDATES if c in ds.column_names), None)
assert src_key is not None, f"❌ لم أجد أي عمود مصدر من: {SRC_CANDIDATES}"

def tok_batch(batch):
    # batch: dict of lists
    sources = [ensure_prefix(clean_merged(s)) for s in batch[src_key]]
    model_inputs = tok(sources, truncation=True, max_length=MAX_IN)
    labels = tok(text_target=[_tidy(t) for t in batch[tgt_key]], truncation=True, max_length=MAX_OUT)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# split
spl = ds.train_test_split(test_size=0.10, seed=SEED)
tr, va = spl["train"], spl["test"]

# map
tr = tr.map(tok_batch, batched=True, remove_columns=tr.column_names)
va = va.map(tok_batch, batched=True, remove_columns=va.column_names)

# collator
coll = DataCollatorForSeq2Seq(tokenizer=tok, model=mdl, pad_to_multiple_of=8)

# mixed precision policy
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

# ---------------- Metrics (ROUGE) ----------------
rouge = evaluate.load("rouge")
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    labels = np.where(labels == -100, tok.pad_token_id, labels)
    decoded_preds  = [_tidy(s) for s in tok.batch_decode(preds,  skip_special_tokens=True)]
    decoded_labels = [_tidy(s) for s in tok.batch_decode(labels, skip_special_tokens=True)]
    res = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {k: float(res[k]) for k in res.keys()}

# ---------------- Training Args (نسخ متوافقة تلقائيًا) ----------------
sig_args = signature(Seq2SeqTrainingArguments.__init__).parameters

eval_kw = {}
if "eval_strategy" in sig_args:
    eval_kw["eval_strategy"] = "epoch"
elif "evaluation_strategy" in sig_args:
    eval_kw["evaluation_strategy"] = "epoch"
else:
    # بعض النسخ القديمة جدًا لا تدعم التقييم الآلي؛ نتجاهله بأمان
    pass

args_kwargs = dict(
    output_dir=OUT,
    num_train_epochs=8,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.2,

    optim="adafactor",
    weight_decay=0.01,
    max_grad_norm=1.0,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    eval_accumulation_steps=1,

    predict_with_generate=True,
    generation_num_beams=4,

    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="rougeLsum",
    greater_is_better=True,

    logging_steps=20,
    logging_first_step=True,
    report_to=[],

    fp16=(not use_bf16),
    bf16=use_bf16,
    group_by_length=True,
    dataloader_num_workers=2,
    gradient_checkpointing=True,
    save_safetensors=True,
)
# عالج دعم generation_max_length حسب النسخة
if "generation_max_length" in sig_args:
    args_kwargs["generation_max_length"] = MAX_OUT
else:
    # بديل آمن للنسخ القديمة: ضبط config مباشرةً
    try:
        mdl.generation_config.max_length = MAX_OUT
    except Exception:
        mdl.config.max_length = MAX_OUT

# (اختياري) dataloader_pin_memory موجود تقريبًا دائمًا، لكن نتأكد
if "dataloader_pin_memory" in sig_args:
    args_kwargs["dataloader_pin_memory"] = True

# حقن eval/evaluation_strategy الصحيح
args_kwargs.update(eval_kw)

args = Seq2SeqTrainingArguments(**args_kwargs)

trainer = Seq2SeqTrainer(
    model=mdl,
    args=args,
    train_dataset=tr,
    eval_dataset=va,
    data_collator=coll,
    compute_metrics=compute_metrics,
    callbacks=[],   # ⛔ بدون EarlyStopping
    tokenizer=tok,
)

print(f"Device: {device} | bf16={args.bf16} | fp16={args.fp16}")
print("Source column:", src_key, "| Target column:", tgt_key)
print("Effective batch (per_device x accum):",
      f"{args.per_device_train_batch_size} x {args.gradient_accumulation_steps}")

# ---------------- Train ----------------
trainer.train()  # يكمل 8 عصور كاملة

# ---------------- Save BEST ----------------
best_dir = f"{OUT}/checkpoints/final_best_{RUN_NAME}"
trainer.save_model(best_dir)

# ثبّت إعدادات التوليد مع IDs الحرِجة لتفادي أخطاء لاحقة
gen_cfg = GenerationConfig(
    num_beams=4, do_sample=False, max_new_tokens=MAX_OUT, early_stopping=True,
    decoder_start_token_id=mdl.config.decoder_start_token_id,
    pad_token_id=mdl.config.pad_token_id,
    bos_token_id=getattr(mdl.config, "bos_token_id", None) or mdl.config.pad_token_id,
    eos_token_id=getattr(mdl.config, "eos_token_id", None) or tok.eos_token_id or mdl.config.pad_token_id,
)
gen_cfg.save_pretrained(best_dir)
tok.save_pretrained(best_dir)

print("✅ Saved BEST ->", best_dir)

# ---------------- Cleanup ----------------
del tr, va, ds, df, recs
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# === Generate predictions from BEST checkpoint + pick Top-5 by ROUGE-Lsum (best-over-refs) ===
import os, json, re, math, glob, torch, pandas as pd, numpy as np
from datetime import datetime
from typing import List, Dict, Any
from tqdm.auto import tqdm
import evaluate
from transformers import T5TokenizerFast, T5ForConditionalGeneration, AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig

# -------- Run paths --------
try:
    RUN_NAME
except NameError:
    RUN_NAME = "arat5-base-20251009-0826"  # عدّلها إذا لزم

RUN_DIR  = f"/content/drive/MyDrive/sum_runs/{RUN_NAME}"
BEST_DIR = f"{RUN_DIR}/checkpoints/final_best_{RUN_NAME}"
PRED_DIR = f"{RUN_DIR}/preds"
OUT_DIR  = f"{RUN_DIR}/metrics"
os.makedirs(PRED_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"  # عدّل المسار لو مختلف
AR_PREFIX = "لخص بالعربية الفصحى بدقة واختصار ووضوح:"
MAX_IN, MAX_OUT = 6144, 512
device = "cuda" if torch.cuda.is_available() else "cpu"

# -------- Helpers --------
CTRL_PREFIX_RE = re.compile(r"^\s*(?:ل?\s?خص)\s*بالعربية\s+الفصحى\s+بدقة\s+واختصار\s+ووضوح\s*:\s*", re.IGNORECASE)
TAG_RE = re.compile(r"</?ARTICLE_\d+>", re.IGNORECASE)

def sanitize_text(s: str) -> str:
    if not isinstance(s, str): s = "" if s is None else str(s)
    s = TAG_RE.sub(" ", s)
    s = CTRL_PREFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    return s

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    recs = []
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        for raw in f:
            s = raw.strip()
            if not s or s.startswith("```"): continue
            if s.startswith("`"): s = s[1:]
            s = s.rstrip("`,")
            try:
                obj = json.loads(s)
                if isinstance(obj, dict): recs.append(obj)
            except json.JSONDecodeError:
                pass
    return recs

def get_source(rec: Dict[str, Any]) -> str:
    return (rec.get("Merged_Articles") or rec.get("merged_articles") or
            rec.get("model_input") or rec.get("text") or "")

# -------- Load tokenizer/model safely --------
def load_t5_or_auto(ckpt_dir: str):
    try:
        tok = T5TokenizerFast.from_pretrained(ckpt_dir, use_fast=True)
        mdl = T5ForConditionalGeneration.from_pretrained(ckpt_dir)
    except Exception:
        cfg = AutoConfig.from_pretrained(ckpt_dir)
        tok = AutoTokenizer.from_pretrained(ckpt_dir, use_fast=True)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(ckpt_dir, config=cfg)
    # IDs الضرورية
    if getattr(mdl.config, "pad_token_id", None) is None:
        mdl.config.pad_token_id = tok.pad_token_id
    if getattr(mdl.config, "decoder_start_token_id", None) is None:
        mdl.config.decoder_start_token_id = tok.pad_token_id
    # (اختياري) ثبّت bos/eos لو ناقصين
    if getattr(mdl.config, "bos_token_id", None) is None:
        mdl.config.bos_token_id = mdl.config.pad_token_id
    if getattr(mdl.config, "eos_token_id", None) is None:
        mdl.config.eos_token_id = getattr(tok, "eos_token_id", None) or mdl.config.pad_token_id
    return tok, mdl

tok, mdl = load_t5_or_auto(BEST_DIR)
mdl = mdl.to(device).eval()

@torch.no_grad()
def generate_one(txt: str) -> str:
    x = tok(AR_PREFIX + " " + (txt or ""), return_tensors="pt",
            truncation=True, max_length=MAX_IN).to(device)
    out = mdl.generate(
        **x,
        num_beams=4,
        length_penalty=1.0,
        no_repeat_ngram_size=3,
        min_new_tokens=80,
        max_new_tokens=MAX_OUT,
        decoder_start_token_id=mdl.config.decoder_start_token_id,
        pad_token_id=mdl.config.pad_token_id,
        bos_token_id=mdl.config.bos_token_id,
        eos_token_id=mdl.config.eos_token_id,
    )
    return tok.decode(out[0], skip_special_tokens=True)

# -------- Load test & generate --------
items = load_jsonl(TEST_PATH)
assert items, f"❌ لا توجد عناصر في {TEST_PATH}"

preds, rows = [], []
for i, rec in tqdm(list(enumerate(items)), desc="Generating"):
    gen = sanitize_text(generate_one(get_source(rec)))
    r1 = sanitize_text(rec.get("target_summary", "") or "")
    r2 = sanitize_text(rec.get("target_summary_2", "") or "")
    rows.append({
        "id": rec.get("id", i),
        "title": rec.get("title", ""),
        "prediction": gen,
        "ref1": r1,
        "ref2": r2,
    })
    preds.append(gen)

# -------- Per-sample ROUGE-Lsum best-over-refs --------
rouge = evaluate.load("rouge")
rougeLsum_best, rougeLsum_r1, rougeLsum_r2, best_ref = [], [], [], []

for i, row in enumerate(rows):
    p  = row["prediction"]
    r1 = row["ref1"] or ""
    r2 = row["ref2"] or ""
    sc1 = rouge.compute(predictions=[p], references=[r1], use_stemmer=False)["rougeLsum"] if r1 else 0.0
    sc2 = rouge.compute(predictions=[p], references=[r2], use_stemmer=False)["rougeLsum"] if r2 else 0.0
    rougeLsum_r1.append(float(sc1))
    rougeLsum_r2.append(float(sc2))
    if sc2 > sc1:
        rougeLsum_best.append(float(sc2)); best_ref.append("Ref2")
    else:
        rougeLsum_best.append(float(sc1)); best_ref.append("Ref1")

for i, row in enumerate(rows):
    row["rougeLsum_ref1"] = rougeLsum_r1[i]
    row["rougeLsum_ref2"] = rougeLsum_r2[i]
    row["rougeLsum_best"] = rougeLsum_best[i]
    row["best_ref"]       = best_ref[i]

# -------- Save full predictions + Top-5 --------
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
pred_path = f"{PRED_DIR}/preds_{RUN_NAME}_{ts}.csv"
pd.DataFrame(rows).to_csv(pred_path, index=False, encoding="utf-8-sig")

top5 = sorted(rows, key=lambda r: r["rougeLsum_best"], reverse=True)[:5]
top5_path = f"{OUT_DIR}/top5_{RUN_NAME}_{ts}.csv"
pd.DataFrame(top5).to_csv(top5_path, index=False, encoding="utf-8-sig")

print("✅ Saved predictions ->", pred_path)
print("✅ Saved Top-5 ->", top5_path)

# -------- Pretty print Top-5 (generated + both refs) --------
print("\n====== Top-5 by ROUGE-Lsum (best-over-refs) ======")
for k, r in enumerate(top5, 1):
    print(f"\n#{k} | id={r['id']} | best_ref={r['best_ref']} | rougeLsum_best={r['rougeLsum_best']:.4f}")
    if r.get("title"): print("• العنوان:", r["title"])
    print("• المرجع 1:", r["ref1"] if r["ref1"] else "—")
    print("• المرجع 2:", r["ref2"] if r["ref2"] else "—")
    print("• المولَّد :", r["prediction"])


In [ ]:
# === Arabic-constrained Generation + Top-5 (with both references) ===
import os, re, json, glob, math, torch, pandas as pd, numpy as np
from datetime import datetime
from typing import List, Dict, Any
from tqdm.auto import tqdm
import evaluate
from transformers import T5TokenizerFast, T5ForConditionalGeneration, AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig

# ---------- Paths ----------
try:
    RUN_NAME
except NameError:
    RUN_NAME = "arat5-base-20251009-0828"  # عدّلها لو لزم

RUN_DIR  = f"/content/drive/MyDrive/sum_runs/{RUN_NAME}"
BEST_DIR = f"{RUN_DIR}/checkpoints/final_best_{RUN_NAME}"
PRED_DIR = f"{RUN_DIR}/preds"
OUT_DIR  = f"{RUN_DIR}/metrics"
os.makedirs(PRED_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"
AR_PREFIX = "لخص بالعربية الفصحى بدقة واختصار ووضوح:"
MAX_IN, MAX_OUT = 3072, 256   # تقليل الطول يساعد T5-base على الاستقرار
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- Load test ----------
def load_jsonl(path: str):
    recs=[]
    with open(path,"r",encoding="utf-8-sig",errors="replace") as f:
        for ln in f:
            s=ln.strip()
            if not s: continue
            try: recs.append(json.loads(s))
            except: pass
    return recs

def get_source(rec: Dict[str,Any]) -> str:
    return (rec.get("Merged_Articles") or rec.get("merged_articles") or
            rec.get("model_input") or rec.get("text") or "")

items = load_jsonl(TEST_PATH)
assert items, f"❌ لا توجد عناصر في {TEST_PATH}"

# ---------- Sanitization ----------
CTRL_PREFIX_RE = re.compile(r"^\s*(?:ل?\s?خص)\s*بالعربية\s+الفصحى\s+بدقة\s+واختصار\s+ووضوح\s*:\s*", re.IGNORECASE)
TAG_RE = re.compile(r"</?ARTICLE_\d+>", re.IGNORECASE)
def sanitize_text(s: str) -> str:
    if not isinstance(s, str): s = "" if s is None else str(s)
    s = TAG_RE.sub(" ", s)
    s = CTRL_PREFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    return s

# ---------- Load tokenizer/model ----------
def load_t5_or_auto(ckpt_dir: str):
    try:
        tok = T5TokenizerFast.from_pretrained(ckpt_dir, use_fast=True)
        mdl = T5ForConditionalGeneration.from_pretrained(ckpt_dir)
    except Exception:
        cfg = AutoConfig.from_pretrained(ckpt_dir)
        tok = AutoTokenizer.from_pretrained(ckpt_dir, use_fast=True)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(ckpt_dir, config=cfg)
    # ensure critical IDs
    if getattr(mdl.config, "pad_token_id", None) is None:
        mdl.config.pad_token_id = tok.pad_token_id
    if getattr(mdl.config, "decoder_start_token_id", None) is None:
        mdl.config.decoder_start_token_id = tok.pad_token_id
    if getattr(mdl.config, "bos_token_id", None) is None:
        mdl.config.bos_token_id = mdl.config.pad_token_id
    if getattr(mdl.config, "eos_token_id", None) is None:
        mdl.config.eos_token_id = getattr(tok, "eos_token_id", None) or mdl.config.pad_token_id
    return tok, mdl

tok, mdl = load_t5_or_auto(BEST_DIR)
mdl = mdl.to(device).eval()

# ---------- Build Arabic-only constraints ----------
# نسمح بالعربية + الفراغ/النقاط/الفواصل وبعض الرموز العربية؛ ونحظر الباقي
import unicodedata as ud
ALLOWED_EXTRA = set(list(".,،؛:؟!()[]{}«»\"'…-–—%+*/=  \t\n\r"))
def is_arabic_char(ch: str) -> bool:
    cp = ord(ch)
    return (
        0x0600 <= cp <= 0x06FF or  # Arabic
        0x0750 <= cp <= 0x077F or  # Arabic Supplement
        0x08A0 <= cp <= 0x08FF or  # Arabic Extended-A
        0xFB50 <= cp <= 0xFDFF or  # Arabic Presentation Forms-A
        0xFE70 <= cp <= 0xFEFF or  # Arabic Presentation Forms-B
        ch == "▁" or ch in ALLOWED_EXTRA
    )

bad_ids = []
for tok_str, tok_id in tok.get_vocab().items():
    # استثنِ التوكنات الخاصة
    if tok_str in ["<pad>", "</s>", "<unk>"]:
        continue
    # إن احتوى على حرف غير عربي/غير مسموح → ضعه في قائمة المحظورات
    if any(not is_arabic_char(c) for c in tok_str):
        bad_ids.append(tok_id)

# bad_words_ids يجب أن تكون قائمة قوائم IDs
bad_words_ids = [[i] for i in sorted(set(bad_ids)) if i not in {tok.pad_token_id, getattr(tok,"eos_token_id", -1)}]

# ---------- Generate ----------
@torch.no_grad()
def generate_one(txt: str) -> str:
    x = tok(AR_PREFIX + " " + (txt or ""), return_tensors="pt",
            truncation=True, max_length=MAX_IN).to(device)
    out = mdl.generate(
        **x,
        num_beams=4,
        length_penalty=1.05,
        no_repeat_ngram_size=4,
        repetition_penalty=1.15,
        min_new_tokens=40,
        max_new_tokens=MAX_OUT,
        bad_words_ids=bad_words_ids,  # ⛔ امنع الحروف/الإيموجي غير العربية
        decoder_start_token_id=mdl.config.decoder_start_token_id,
        pad_token_id=mdl.config.pad_token_id,
        bos_token_id=mdl.config.bos_token_id,
        eos_token_id=mdl.config.eos_token_id,
    )
    return tok.decode(out[0], skip_special_tokens=True)

rows = []
for i, rec in tqdm(list(enumerate(items)), desc="Generating (Arabic-constrained)"):
    gen = sanitize_text(generate_one(get_source(rec)))
    r1 = sanitize_text(rec.get("target_summary", "") or "")
    r2 = sanitize_text(rec.get("target_summary_2", "") or "")
    rows.append({
        "id": rec.get("id", i),
        "title": rec.get("title", ""),
        "prediction": gen,
        "ref1": r1,
        "ref2": r2,
    })

# ---------- Score & Top-5 ----------
rouge = evaluate.load("rouge")
for r in rows:
    p, r1, r2 = r["prediction"], r["ref1"], r["ref2"]
    sc1 = rouge.compute(predictions=[p], references=[r1], use_stemmer=False)["rougeLsum"] if r1 else 0.0
    sc2 = rouge.compute(predictions=[p], references=[r2], use_stemmer=False)["rougeLsum"] if r2 else 0.0
    r["rougeLsum_ref1"] = float(sc1)
    r["rougeLsum_ref2"] = float(sc2)
    r["rougeLsum_best"] = float(max(sc1, sc2))
    r["best_ref"] = "Ref1" if sc1 >= sc2 else "Ref2"

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
pred_path = f"{PRED_DIR}/preds_{RUN_NAME}_{ts}_ARONLY.csv"
pd.DataFrame(rows).to_csv(pred_path, index=False, encoding="utf-8-sig")

top5 = sorted(rows, key=lambda r: r["rougeLsum_best"], reverse=True)[:5]
top5_path = f"{OUT_DIR}/top5_{RUN_NAME}_{ts}_ARONLY.csv"
pd.DataFrame(top5).to_csv(top5_path, index=False, encoding="utf-8-sig")

print("✅ Saved predictions ->", pred_path)
print("✅ Saved Top-5 ->", top5_path)

print("\n====== Top-5 by ROUGE-Lsum (best-over-refs, Arabic-constrained) ======")
for k, r in enumerate(top5, 1):
    print(f"\n#{k} | id={r['id']} | best_ref={r['best_ref']} | rougeLsum_best={r['rougeLsum_best']:.4f}")
    if r.get("title"): print("• العنوان:", r["title"])
    print("• المرجع 1:", r["ref1"] if r["ref1"] else "—")
    print("• المرجع 2:", r["ref2"] if r["ref2"] else "—")
    print("• المولَّد :", r["prediction"])


In [ ]:
# =================== Comprehensive Evaluation (Ref1 / Ref2 / Avg / Best) — Run-aware & Robust ===================
!pip -q install "pandas>=2.0" "evaluate==0.4.2" "rouge_score==0.1.2" "sacrebleu==2.4.0" \
                 "bert-score==0.3.13" "transformers>=4.41.0" "pot>=0.9.3" tqdm

import os, re, json, math, glob
from typing import List, Dict, Any, Tuple
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModel
import evaluate
from bert_score import BERTScorer
import ot  # POT

# --------- Run-aware paths ---------
try:
    RUN_NAME
except NameError:
    # عدّل هذا إذا لزم
    RUN_NAME = "arat5-base-20251009-0828"

MODEL_FAMILY = "arat5"  # لمعلومات السجل فقط
RUN_DIR  = f"/content/drive/MyDrive/sum_runs/{RUN_NAME}"
PRED_DIR = f"{RUN_DIR}/preds"
CKPT_DIR = f"{RUN_DIR}/checkpoints/final_best_{RUN_NAME}"
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"

# إن أردت تحديد مسار ملف تنبؤات بعينه، ضع المسار هنا:
PRED_PATH_OVERRIDE = None

def _latest_pred(pred_dir: str):
    if not os.path.isdir(pred_dir): return None
    cands = []
    for ext in ("*.csv", "*.jsonl", "*.json"):
        cands.extend(glob.glob(os.path.join(pred_dir, ext)))
    if not cands: return None
    # فضّل ملفات ARONLY إن وُجدت
    ar_cands = [p for p in cands if "_ARONLY" in os.path.basename(p)]
    use = ar_cands if ar_cands else cands
    use.sort(key=os.path.getmtime, reverse=True)
    return use[0]

PRED_PATH = PRED_PATH_OVERRIDE or _latest_pred(PRED_DIR)
assert PRED_PATH and os.path.exists(PRED_PATH), \
    f"❌ لم أجد ملف تنبؤات داخل {PRED_DIR}. حدِّد PRED_PATH_OVERRIDE أو صدِّر تنبؤاتك."

OUT_DIR = f"{RUN_DIR}/metrics/eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT_DIR, exist_ok=True)

print("✅ RUN_NAME:", RUN_NAME)
print("📁 RUN_DIR :", RUN_DIR)
print("📦 CKPT_DIR:", CKPT_DIR)
print("📝 PRED_PATH:", PRED_PATH)
print("📊 OUT_DIR :", OUT_DIR)

# --------- Env ---------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("🚀 Device:", device)

# --------- Helpers ---------
CTRL_PREFIX_RE = re.compile(r"^\s*(?:ل?\s?خص)\s*بالعربية\s+الفصحى\s+بدقة\s+واختصار\s+ووضوح\s*:\s*", re.IGNORECASE)
TAG_RE = re.compile(r"</?ARTICLE_\d+>", re.IGNORECASE)

def sanitize_text(s: str) -> str:
    if not isinstance(s, str): s = "" if s is None else str(s)
    s = TAG_RE.sub(" ", s)
    s = CTRL_PREFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    return s

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    recs = []
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        for raw in f:
            s = raw.strip()
            if not s or s.startswith("```"): continue
            if s.startswith("`"): s = s[1:]
            s = s.rstrip("`,")
            try:
                obj = json.loads(s)
                if isinstance(obj, dict): recs.append(obj)
            except json.JSONDecodeError:
                pass
    return recs

# --------- Load test + refs ---------
items = load_jsonl(TEST_PATH)
assert items, f"❌ لا يوجد عناصر في {TEST_PATH}"
print("🧪 Loaded items:", len(items))

ref1 = [sanitize_text(x.get("target_summary", "")) for x in items]
ref2 = [sanitize_text(x.get("target_summary_2", "")) for x in items]
has_r2 = [bool(r.strip()) for r in ref2]
num_r2 = sum(1 for x in has_r2 if x)
print(f"Ref1 count: {len(ref1)} | Ref2 non-empty: {num_r2}/{len(ref2)}")

# --------- Load predictions ---------
def load_predictions(pred_path: str, test_items: List[Dict[str, Any]]) -> List[str]:
    id2pred = {}; preds_list = []
    if pred_path.lower().endswith(".csv"):
        df = pd.read_csv(pred_path)
        pred_col = None
        for c in ["prediction","pred","output","summary","generated","generated_summary"]:
            if c in df.columns: pred_col = c; break
        if pred_col is None:
            raise ValueError(f"لم أجد عمود التنبؤات في CSV. الأعمدة: {list(df.columns)}")
        if "id" in df.columns:
            for _, row in df.iterrows():
                id2pred[str(row["id"])] = sanitize_text(row[pred_col])
        else:
            preds_list = [sanitize_text(x) for x in df[pred_col].astype(str).tolist()]
    else:
        try:
            with open(pred_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, dict) and "records" in data: data = data["records"]
            if not isinstance(data, list): data = [data]
        except Exception:
            data = load_jsonl(pred_path)
        for obj in data:
            pred_txt = obj.get("prediction") or obj.get("pred") or obj.get("summary") or ""
            pred_txt = sanitize_text(pred_txt)
            if "id" in obj: id2pred[str(obj["id"])] = pred_txt
            else: preds_list.append(pred_txt)

    preds = []
    for i, rec in enumerate(test_items):
        key = str(rec.get("id", i))
        if id2pred:
            preds.append(id2pred.get(key, ""))
        else:
            preds.append(preds_list[i] if i < len(preds_list) else "")
    return preds

preds = load_predictions(PRED_PATH, items)
preds = [sanitize_text(p) for p in preds]
print("✅ Predictions loaded:", len(preds))

# --------- Quick diagnostics ---------
nonempty = sum(1 for p in preds if p.strip())
avg_words = float(np.mean([len(p.split()) for p in preds])) if preds else 0.0
print(f"🔎 Pred diagnostics -> non-empty: {nonempty}/{len(preds)} | avg_words: {avg_words:.1f}")

# --------- Build refs structures ---------
refs_multi = []
for i in range(len(items)):
    rlist = []
    if ref1[i].strip(): rlist.append(ref1[i])
    if ref2[i].strip(): rlist.append(ref2[i])
    if not rlist: rlist = [""]  # ضمان وجود مرجع
    refs_multi.append(rlist)

# =================== ROUGE ===================
rouge = evaluate.load("rouge")
def rouge_scores(pred, ref):
    sc = rouge.compute(predictions=pred, references=ref, use_stemmer=False)
    return {k: float(sc[k]) for k in ["rouge1","rouge2","rougeL","rougeLsum"]}

rouge_ref1 = rouge_scores(preds, ref1)
rouge_ref2 = rouge_scores(preds, [r if r else "" for r in ref2]) if num_r2>0 else {k: None for k in ["rouge1","rouge2","rougeL","rougeLsum"]}

# best-over-refs (ROUGE-Lsum per-sample)
rouge1_list = []; rouge2_list = []; rougeL_list = []; rougeLsum_list = []; best_ref_ix=[]
for i in tqdm(range(len(items)), desc="ROUGE best-over-refs"):
    rs = [x for x in [ref1[i], ref2[i]] if x] or [""]
    best = {"r1":0.0,"r2":0.0,"rL":0.0,"rLsum":0.0}; bix=0
    for j, r in enumerate(rs):
        sc = rouge.compute(predictions=[preds[i]], references=[r], use_stemmer=False)
        if sc["rougeLsum"] > best["rLsum"]:
            best = {"r1":float(sc["rouge1"]), "r2":float(sc["rouge2"]), "rL":float(sc["rougeL"]), "rLsum":float(sc["rougeLsum"])}
            bix=j
    rouge1_list.append(best["r1"]); rouge2_list.append(best["r2"])
    rougeL_list.append(best["rL"]); rougeLsum_list.append(best["rLsum"])
    best_ref_ix.append(bix)
rouge_best = {
    "rouge1": float(np.mean(rouge1_list)),
    "rouge2": float(np.mean(rouge2_list)),
    "rougeL": float(np.mean(rougeL_list)),
    "rougeLsum": float(np.mean(rougeLsum_list)),
}
rouge_avg = {k: None if rouge_ref2[k] is None else (rouge_ref1[k] + rouge_ref2[k]) / 2.0 for k in rouge_ref1}

# =================== SacreBLEU / chrF ===================
sacrebleu = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def wrap_single(refs):  # [[ref_i] ...]
    return [[r] for r in refs]

bleu_multi = sacrebleu.compute(predictions=preds, references=refs_multi)["score"]
bleu_r1    = sacrebleu.compute(predictions=preds, references=wrap_single(ref1))["score"]
bleu_r2    = sacrebleu.compute(predictions=preds, references=wrap_single([r if r else "" for r in ref2]))["score"] if num_r2>0 else None
bleu_avg   = None if bleu_r2 is None else (bleu_r1 + bleu_r2) / 2.0

chrf_multi = chrf_metric.compute(predictions=preds, references=refs_multi)["score"]
chrf_r1    = chrf_metric.compute(predictions=preds, references=wrap_single(ref1))["score"]
chrf_r2    = chrf_metric.compute(predictions=preds, references=wrap_single([r if r else "" for r in ref2]))["score"] if num_r2>0 else None
chrf_avg   = None if chrf_r2 is None else (chrf_r1 + chrf_r2) / 2.0

# =================== METEOR ===================
try:
    meteor_metric = evaluate.load("meteor")
    met_r1 = meteor_metric.compute(predictions=preds, references=ref1)["meteor"]
    met_r2 = meteor_metric.compute(predictions=preds, references=[r if r else "" for r in ref2])["meteor"] if num_r2>0 else None
    # best-over-refs
    met_best_list = []
    for i in tqdm(range(len(items)), desc="METEOR best-over-refs"):
        rs = [x for x in [ref1[i], ref2[i]] if x] or [""]
        best = 0.0
        for r in rs:
            best = max(best, meteor_metric.compute(predictions=[preds[i]], references=[r])["meteor"])
        met_best_list.append(best)
    met_best = float(np.mean(met_best_list))
except Exception as e:
    print("METEOR skipped:", e)
    met_r1 = met_r2 = met_best = None

# =================== BERTScore (direct scorer; FIX: compute_idf before score) ===================
print("\n🎯 BERTScore (per-ref + best) — direct scorer")
BERT_MODEL = "xlm-roberta-large"
scorer = BERTScorer(model_type=BERT_MODEL, lang=None, rescale_with_baseline=False, idf=True, device=device, batch_size=8)

# ✅ FIX: احسب IDF من كل المراجع المتاحة
all_refs_flat = [r for rs in refs_multi for r in rs]
scorer.compute_idf(all_refs_flat)

def bert_over(preds, refs):
    P,R,F = scorer.score(preds, refs)
    return float(torch.mean(P).item()), float(torch.mean(R).item()), float(torch.mean(F).item())

bP1,bR1,bF1 = bert_over(preds, ref1)
if num_r2>0:
    bP2,bR2,bF2 = bert_over(preds, [r if r else "" for r in ref2])
else:
    bP2=bR2=bF2=None

P1,R1,F1 = scorer.score(preds, ref1)
if num_r2>0:
    P2,R2,F2 = scorer.score(preds, [r if r else "" for r in ref2])
bP_best=[]; bR_best=[]; bF_best=[]
for i in range(len(items)):
    if num_r2>0 and F2[i] > F1[i]:
        bP_best.append(float(P2[i])); bR_best.append(float(R2[i])); bF_best.append(float(F2[i]))
    else:
        bP_best.append(float(P1[i])); bR_best.append(float(R1[i])); bF_best.append(float(F1[i]))
bP_best = float(np.mean(bP_best)); bR_best = float(np.mean(bR_best)); bF_best = float(np.mean(bF_best))

# =================== MoverScore (POT/OT) ===================
print("\n⚖️  MoverScore (per-ref + best) — POT/OT")
EMB_MODEL = "xlm-roberta-large"
MAX_LEN   = 512
tok_emb = AutoTokenizer.from_pretrained(EMB_MODEL)
mdl_emb = AutoModel.from_pretrained(EMB_MODEL).to(device).eval()

@torch.no_grad()
def embed_tokens(text: str, max_len: int = MAX_LEN) -> Tuple[torch.Tensor, List[int]]:
    out = tok_emb(text or "", return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=True)
    out = {k: v.to(device) for k, v in out.items()}
    hs = mdl_emb(**out).last_hidden_state.squeeze(0)
    ids = out["input_ids"].squeeze(0).tolist()
    special = set(t for t in [getattr(tok_emb, "cls_token_id", None),
                              getattr(tok_emb, "sep_token_id", None),
                              getattr(tok_emb, "pad_token_id", None)] if t is not None)
    keep = [i for i, tid in enumerate(ids) if tid not in special] or list(range(len(ids)))
    return hs[keep], [ids[i] for i in keep]

def build_idf_from_texts(texts: List[str], max_len: int = MAX_LEN) -> dict:
    from collections import Counter, defaultdict
    df_counter = Counter()
    for t in texts:
        ids = tok_emb(t or "", truncation=True, max_length=max_len, add_special_tokens=True)["input_ids"]
        df_counter.update(set(ids))
    N = max(1, len(texts))
    idf = defaultdict(lambda: 0.0)
    for tid, dfv in df_counter.items():
        idf[tid] = math.log((N + 1) / (dfv + 1))
    return idf

def cosine_cost(X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
    Xn = torch.nn.functional.normalize(X, dim=1)
    Yn = torch.nn.functional.normalize(Y, dim=1)
    sim = Xn @ Yn.T
    return (1.0 - sim).clamp(min=0.0)

idf_pred = build_idf_from_texts(preds)
idf_ref1 = build_idf_from_texts(ref1)
idf_ref2 = build_idf_from_texts([r if r else "" for r in ref2]) if num_r2>0 else {}

def moverscore_pair(hyp: str, ref: str, idf_h: dict, idf_r: dict) -> float:
    H, ids_h = embed_tokens(hyp, MAX_LEN)
    R, ids_r = embed_tokens(ref, MAX_LEN)
    if H.shape[0] == 0 or R.shape[0] == 0: return 0.0
    a = torch.tensor([idf_h.get(tid, 0.0) for tid in ids_h], dtype=torch.float32)
    b = torch.tensor([idf_r.get(tid, 0.0) for tid in ids_r], dtype=torch.float32)
    if float(a.sum()) == 0: a = torch.ones_like(a)
    if float(b.sum()) == 0: b = torch.ones_like(b)
    a = (a / a.sum()).cpu().numpy()
    b = (b / b.sum()).cpu().numpy()
    C = cosine_cost(H, R).cpu().numpy().astype(np.float64)
    T = ot.emd(a, b, C)
    cost = float((T * C).sum())
    return max(0.0, min(1.0, 1.0 - cost))

ms_r1, ms_r2, ms_best = [], [], []
for i in tqdm(range(len(items)), desc="MoverScore"):
    p = preds[i]
    r1 = ref1[i] if ref1[i] else ""
    s1 = moverscore_pair(p, r1, idf_pred, idf_ref1)
    ms_r1.append(s1)
    if num_r2>0 and ref2[i]:
        s2 = moverscore_pair(p, ref2[i], idf_pred, idf_ref2)
        ms_r2.append(s2)
        ms_best.append(max(s1, s2))
    else:
        ms_best.append(s1)

mover_ref1 = float(np.mean(ms_r1))
mover_ref2 = float(np.mean(ms_r2)) if num_r2>0 and ms_r2 else None
mover_best = float(np.mean(ms_best))

# =================== Summary & Save ===================
summary = {
    "run_name": RUN_NAME,
    "model_family": MODEL_FAMILY,
    "run_dir": RUN_DIR,
    "ckpt_dir": CKPT_DIR,
    "count": len(items),
    "pred_non_empty": nonempty,
    "pred_avg_words": avg_words,
    # ROUGE
    "rouge1_ref1_mean": rouge_ref1["rouge1"],
    "rouge2_ref1_mean": rouge_ref1["rouge2"],
    "rougeL_ref1_mean": rouge_ref1["rougeL"],
    "rougeLsum_ref1_mean": rouge_ref1["rougeLsum"],
    "rouge1_ref2_mean": rouge_ref2["rouge1"] if rouge_ref2["rouge1"] is not None else None,
    "rouge2_ref2_mean": rouge_ref2["rouge2"] if rouge_ref2["rouge2"] is not None else None,
    "rougeL_ref2_mean": rouge_ref2["rougeL"] if rouge_ref2["rougeL"] is not None else None,
    "rougeLsum_ref2_mean": rouge_ref2["rougeLsum"] if rouge_ref2["rougeLsum"] is not None else None,
    "rouge1_best_over_refs": float(np.mean(rouge1_list)),
    "rouge2_best_over_refs": float(np.mean(rouge2_list)),
    "rougeL_best_over_refs": float(np.mean(rougeL_list)),
    "rougeLsum_best_over_refs": float(np.mean(rougeLsum_list)),
    # BLEU / chrF
    "sacrebleu_multi_ref": bleu_multi,
    "sacrebleu_ref1": bleu_r1,
    "sacrebleu_ref2": bleu_r2,
    "sacrebleu_avg": None if bleu_r2 is None else (bleu_r1 + bleu_r2) / 2.0,
    "chrf_multi_ref": chrf_multi,
    "chrf_ref1": chrf_r1,
    "chrf_ref2": chrf_r2,
    "chrf_avg": None if chrf_r2 is None else (chrf_r1 + chrf_r2) / 2.0,
    # METEOR
    "meteor_ref1": met_r1,
    "meteor_ref2": met_r2,
    "meteor_best_over_refs": None if 'met_best' not in locals() else met_best,
    # BERTScore
    "bertscore_ref1": {"P": bP1, "R": bR1, "F1": bF1},
    "bertscore_ref2": ({"P": bP2, "R": bR2, "F1": bF2} if bF2 is not None else None),
    "bertscore_best_over_refs": {"P": bP_best, "R": bR_best, "F1": bF_best},
    # MoverScore
    "moverscore_ref1": mover_ref1,
    "moverscore_ref2": mover_ref2,
    "moverscore_best_over_refs": mover_best,
    # Paths for provenance
    "preds_path": PRED_PATH,
    "refs_path": TEST_PATH,
}

rows = []
for i in range(len(items)):
    row = {
        "id": items[i].get("id", i),
        "title": items[i].get("title", ""),
        "prediction": preds[i],
        "ref1": ref1[i],
        "ref2": ref2[i],
        "rougeLsum_ref1": rouge.compute(predictions=[preds[i]], references=[ref1[i] or ""], use_stemmer=False)["rougeLsum"] if ref1[i] else 0.0,
        "rougeLsum_ref2": rouge.compute(predictions=[preds[i]], references=[ref2[i] or ""], use_stemmer=False)["rougeLsum"] if ref2[i] else 0.0,
        "rougeLsum_best": rougeLsum_list[i],
        "best_ref": "Ref1" if (best_ref_ix[i]==0 or not ref2[i]) else "Ref2",
        "moverscore_best": ms_best[i],
    }
    rows.append(row)

df = pd.DataFrame(rows)
csv_path = os.path.join(OUT_DIR, "eval_report_per_sample.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
with open(os.path.join(OUT_DIR, "eval_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n=== Overall Metrics ===")
for k,v in summary.items():
    if k in ["preds_path","refs_path","count","run_name","model_family","run_dir","ckpt_dir"]:
        continue
    print(f"{k}: {v}")

print(f"\n📄 Saved per-sample CSV -> {csv_path}")
print(f"📊 Saved summary JSON   -> {os.path.join(OUT_DIR, 'eval_summary.json')}")
print("✅ Done.")
